# Electricity Maps API — carbon intensity ingestion


In [1]:
import os
from dotenv import load_dotenv
import time
import requests
import psycopg2
from datetime import datetime, timezone, timedelta
import pandas as pd

In [2]:
# ── Config ────────────────────────────────────────────────────────
load_dotenv()

EM_API_KEY = os.environ["EM_API_KEY"]
DB_URL     = os.environ["DATABASE_URL"]  

# Electricity Maps zone codes → our country codes
ZONES = {
    "DE": "DE_LU",
    "FR": "FR",
    "ES": "ES",
    "IT": "IT",
    "GB": "GB",
}

BASE_URL = "https://api.electricitymap.org/v3"
HEADERS  = {"auth-token": EM_API_KEY}

In [3]:
# ── Helpers ───────────────────────────────────────────────────────
def insert_rows(conn, rows: list[dict]):
    with conn.cursor() as cur:
        cur.executemany(
            """
            INSERT INTO carbon_intensity (country, timestamp_utc, gco2_per_kwh, method, is_estimated)
            VALUES (%(country)s, %(timestamp_utc)s, %(gco2_per_kwh)s, 'electricitymaps', %(is_estimated)s)
            ON CONFLICT (country, timestamp_utc, method) DO UPDATE
                SET gco2_per_kwh = EXCLUDED.gco2_per_kwh,
                    is_estimated = EXCLUDED.is_estimated
            """,
            rows,
        )
    conn.commit()


def pull_past_range(zone, country_code, conn, start_date, end_date):
    """
    Fetch historical carbon intensity data (gCO2eq/kWh) for a specific zone over a date range.
    
    ElectricityMaps API Limits:
    - Maximum range per request for 'hourly' granularity is 10 days (240 hours).
    - Maximum range per request for 'daily' granularity is 100 days.
    
    This function batches the requests into 10-day increments to satisfy the hourly limit.
    """
    current = start_date
    while current < end_date:
        # Batch the request to a maximum of 10 days to comply with API constraints
        batch_end = min(current + timedelta(days=10), end_date)
        url = "https://api.electricitymap.org/v4/carbon-intensity/past-range"
        params = {
            "zone":  zone,
            "start": current.isoformat(),
            "end":   batch_end.isoformat(),
            "temporalGranularity": "hourly",
            "emissionFactorType":  "lifecycle",
        }
        resp = requests.get(url, headers=HEADERS, params=params)
        resp.raise_for_status()

        # Parse the response and filter out missing data points
        rows = [
            {
                "country":       country_code,
                "timestamp_utc": entry["datetime"],
                "gco2_per_kwh":  entry["carbonIntensity"],
                "is_estimated":  entry.get("isEstimated", False),
            }
            for entry in resp.json().get("data", [])
            if entry.get("carbonIntensity") is not None
        ]

        # Bulk insert the current batch into the database
        insert_rows(conn, rows)

        # Move the cursor forward to the end of the current batch
        current = batch_end
        time.sleep(1)

In [4]:
# ── Main ──────────────────────────────────────────────────────────

def main():
    conn = psycopg2.connect(DB_URL)
    print(f"Connected to DB. Starting pull at {datetime.now(timezone.utc).isoformat()}\n")

    end_date   = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
    start_date = end_date - timedelta(days=90)

    for zone, country_code in ZONES.items():
        print(f"  Pulling {zone} ({country_code})...", end=" ")
        try:
            pull_past_range(zone, country_code, conn, start_date, end_date)

            with conn.cursor() as cur:
                cur.execute(
                    "SELECT COUNT(*) FROM carbon_intensity WHERE country = %s AND method = 'electricitymaps'",
                    (country_code,)
                )
                count = cur.fetchone()[0]
            print(f"  → {count} total rows in DB for {country_code}")

        except requests.HTTPError as e:
            print(f"HTTP {e.response.status_code} — skipping. "
                  f"Log: {e.response.text[:120]}")
            with conn.cursor() as cur:
                cur.execute(
                    "INSERT INTO data_notes (country, note_type, description) "
                    "VALUES (%s, 'gap', %s)",
                    (country_code, f"EM API error on pull: {e.response.status_code}"),
                )
            conn.commit()

        time.sleep(1)  # stay well under 50 req/hr rate limit

    conn.close()
    print("\nDone.")


if __name__ == "__main__":
    main()

Connected to DB. Starting pull at 2026-06-09T10:15:28.044034+00:00

  Pulling DE (DE_LU)...   → 2160 total rows in DB for DE_LU
  Pulling FR (FR)...   → 2160 total rows in DB for FR
  Pulling ES (ES)...   → 2160 total rows in DB for ES
  Pulling IT (IT)...   → 2160 total rows in DB for IT
  Pulling GB (GB)...   → 2160 total rows in DB for GB

Done.
